## Initialisatie


### Imports


In [ ]:
import os
import ee
import pandas as pd
from dotenv import load_dotenv

from tools.Input_data.satellites import sentinel2_harmonized
from tools.Input_data.locaties import select_naturenreserves, RegionMode
from tools.Input_data.Polygons import load_features
from tools.extract_downloads.ee import estimate_extraction_workload ,build_imagecollection_stats, export_featurecollection_to_local, export_featurecollection_to_drive
from tools.data_cleansing.sentinel2 import add_local_cloud_cover, scl_cloud_removal_20m, scale_reflectance, s2cloudless_cloud_removal, cloudscoreplus_cs_removal, cloudscoreplus_cdf_removal
from tools.calculate_spectral_indices.sentinel2 import (
    calc_bsi,
    calc_cire, 
    calc_evi ,
    calc_evi2,
    calc_fai,
    calc_gci,
    calc_gndvi,
    calc_ibi,
    calc_mndwi,
    calc_msi,
    calc_nbr,
    calc_nbr2,
    calc_ndbi,
    calc_ndmi,
    calc_ndre_b5,
    calc_ndre_b6,
    calc_ndre_b7,
    calc_ndsi,
    calc_ndti,
    calc_ndvi,
    calc_ndwi,
    calc_nirv,
    calc_osavi,
    calc_s2wi,
    calc_savi,
    calc_snow_brightness,
    calc_vari,
)


### Notebook project instellen


In [ ]:
load_dotenv()

project_code_ee = os.getenv("PROJECT_CODE_EE")

if not project_code_ee:
    raise RuntimeError("PROJECT_CODE_EE not found")

try:
    ee.Initialize(project=project_code_ee)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=project_code_ee)

In [ ]:
pd.set_option("display.max_rows", 250)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

### Standaard waarden project


In [ ]:
# Projectgebied
natuurgebieden = select_naturenreserves(["Nieuwkoopse Plassen & de Haeck"])
type_coverage: RegionMode = "bounds"
buffer_size_meters: int = 0

filter_date_start: str = "2026-01-01"
filter_date_end: str = "2027-07-12"
local_cloud_cover_threshold: int = 30
cloud_cover_bloat: int = 100

# onderzoekslocaties
inputdata_cord_sys: str = r"EPSG:28992"
inputdata_polygon: str = r"C:\Users\BaSj\Downloads\Habitattypes Sjoerd\Habitattypes Sjoerd\H3150_2025\H3150_2025.shp"

bands_to_revieuw : list[str] = ["NDVI"]
scale_output_raster: int = 10
feature_columns_keep=[]
image_columns_keep=["MGRS_TILE"]

## code


### Onderzoeksgebieden plotten


In [ ]:
input_data_feat_class = load_features(inputdata_polygon, feature_columns_keep)

### Satelliet database opzetten


In [ ]:
s2 = (
    ee.ImageCollection(sentinel2_harmonized)
    .filterDate(filter_date_start, filter_date_end)
    .filterBounds(natuurgebieden)
    # .map(add_local_cloud_cover(natuurgebieden, coverage=type_coverage))
    # .map(scl_cloud_removal_20m(remove_unclassified=False, bloat_n_meters=cloud_cover_bloat))    
    .map(cloudscoreplus_cs_removal(0.6, 0))
    .map(scale_reflectance)
    .map(calc_ndvi)
)

### Kosten schatten

In [ ]:
estimate_extraction_workload(
    feature_collection = input_data_feat_class,
    image_collection= s2,
    band_names= bands_to_revieuw,
    scale= scale_output_raster,
    include_mean= True,
    include_median= True,
    include_percentiles= (25, 75),
    include_stddev= True,
    include_minmax = True,
    include_count= True,
)

### Resultaten berekenen

In [ ]:
resultaten = build_imagecollection_stats(
    feature_collection= input_data_feat_class,
    image_collection= s2,
    band_names= bands_to_revieuw,
    scale= scale_output_raster,
    crs= "EPSG:28992",
    
    feature_property_columns=feature_columns_keep,
    image_property_columns=image_columns_keep,

    include_count=True,
    include_date=True,

    include_mean= True,
    include_median= True,
    include_percentiles = (25, 75),
    include_stddev = True,
    include_minmax = False,

)
resultaten["workload_estimate"]

In [ ]:
# output_df = export_featurecollection_to_local(
#     feature_collection=resultaten["result_feature_collection"],
#     local_file_path=r"output/resultaten.csv",
#     keep_geometry=False,
#     file_format="csv"
# )

In [ ]:
# output_drive = export_featurecollection_to_drive(
#     feature_collection=resultaten["result_feature_collection"],
#     description="testrun",
#     file_name_prefix="OutputresultatenSentinel2",
#     folder="Remote_Sensing",
#     file_format="CSV",
#     selectors=resultaten["selectors"],
#     include_geometry=False
# )